# MicroLive Pipeline Benchmark

This notebook benchmarks the performance of the MicroLive analysis pipeline, measuring the execution time of each major step: data loading, photobleaching correction, cell segmentation, particle tracking, and MSD analysis.

In [1]:
"""
MicroLive Notebook
==================
This notebook requires MicroLive to be installed:
    pip install microlive

For development mode:
    pip install -e /path/to/microlive
"""
# MicroLive imports
from microlive import microscopy as mi
from microlive.utils.device import check_gpu_status

# Verify GPU support
check_gpu_status()

# Standard scientific imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


System Information:
OS: Darwin 24.6.0
CPU: arm
Python: 3.10.19
RAM: 36.00 GB


## 1. Data Loading and Preparation

In [2]:
step_name = "Data Loading"
start_time = time.time()

# Parameters
pixel_size_xy_nm = 130
voxel_size_z_nm = 500
time_interval_sec = 1.0
list_voxels = [voxel_size_z_nm, pixel_size_xy_nm]

# File Path
file_path_str = "/Users/nzlab-la/Desktop/Github/virtual_cell/spatio_temporal_models/results_simulation/simulated_microscopy.tif"
file_path = Path(file_path_str)

image = None

if file_path.exists():
    print(f"Loading data from: {file_path}")
    try:
        # Load TIFF
        image_data = tifffile.imread(file_path)
        
        # Check dimensions and transpose if necessary
        print(f"Original shape: {image_data.shape}")
        
        if len(image_data.shape) == 5:
             # Assume TCZYX and convert to TZYXC
             # T, C, Z, Y, X -> 0, 1, 2, 3, 4
             # Target: T, Z, Y, X, C -> 0, 2, 3, 4, 1
             image = np.transpose(image_data, (0, 2, 3, 4, 1))
        else:
             print("Unexpected dimensions. Using synthetic fallback.")
             image = None
             
    except Exception as e:
        print(f"Error loading file: {e}")
        image = None
else:
    print("File not found. Generating synthetic data...")

if image is None:
    # Generate reproducible synthetic data
    # Format: [T, Z, Y, X, C]
    T, Z, Y, X, C = 30, 5, 256, 256, 2
    print(f"Generating sythetic data with shape: {(T, Z, Y, X, C)}")
    image = np.zeros((T, Z, Y, X, C), dtype=np.uint16)
    
    # Create a simple cell-like blob in the center
    yy, xx = np.meshgrid(np.arange(Y), np.arange(X))
    center_y, center_x = Y // 2, X // 2
    mask = ((yy - center_y)**2 + (xx - center_x)**2) < (Y // 3)**2
    
    for t in range(T):
        for z in range(Z):
            # Cytosol channel
            image[t, z, :, :, 0] = mask * 1000 + np.random.randint(0, 100, (Y, X))
            
            # Particles (Moving spots) in channel 1
            for i in range(5):
                py = center_y + int(20 * np.sin(t/5 + i))
                px = center_x + int(20 * np.cos(t/5 + i))
                if 0 <= py < Y and 0 <= px < X:
                    image[t, z, py, px, 1] = 5000

# Ensure data is uint16
image = image.astype(np.uint16)
print(f"Final Image Shape (TZYXC): {image.shape}")

log_time(step_name, start_time)

Loading data from: /Users/nzlab-la/Desktop/Github/virtual_cell/spatio_temporal_models/results_simulation/simulated_microscopy.tif
Original shape: (360, 4, 10, 512, 512)
Final Image Shape (TZYXC): (360, 10, 512, 512, 4)
[Data Loading] Execution time: 9.6535 seconds


9.65349793434143

## 2. Photobleaching Correction

In [3]:
step_name = "Photobleaching Correction"
start_time = time.time()

# Create mask from first frame max projection
ref_image = image[0, :, :, :, 0].max(axis=0)
threshold = np.mean(ref_image)
mask_YX = (ref_image > threshold).astype(np.uint8)

# Initialize Photobleaching corrector
# Updated based on src/microscopy.py definition
corrector = mi.Photobleaching(
    image_TZYXC=image,
    mask_YX=mask_YX,
    show_plot=False,
    time_interval_seconds=time_interval_sec,
    mode='inside_cell' 
)

# Calculate Correction
# Updated to use apply_photobleaching_correction which returns (image, stats)
image_corrected, _ = corrector.apply_photobleaching_correction()

# Ensure uint16
image_corrected = image_corrected.astype(np.uint16)

log_time(step_name, start_time)
print(f"Corrected image shape: {image_corrected.shape}")

[Photobleaching Correction] Execution time: 15.4873 seconds
Corrected image shape: (360, 10, 512, 512, 4)


## 3. Cell Segmentation

In [4]:
step_name = "Cell Segmentation"
start_time = time.time()

# Input: 2D max projection of first timepoint, channel 0
segmentation_input = image_corrected[0, :, :, :, 0].max(axis=0)

# Initialize Segmentator
# Updated arguments based on source code inspection
segmentator = mi.CellSegmentationWatershed(
    image=segmentation_input,
    expected_radius=30, # approximate for typical cells in px
    min_object_size=100,
    threshold_method='li',
    separation_size=1
)

# Apply Watershed
# Note: apply_watershed returns a single mask (best_mask) in current implementation
mask_cytosol = segmentator.apply_watershed()
mask_nuclei = np.zeros_like(mask_cytosol) # Placeholder if not segmented separately

log_time(step_name, start_time)
print(f"Segmentation complete. Mask shape: {mask_cytosol.shape}")

[Cell Segmentation] Execution time: 0.1372 seconds
Segmentation complete. Mask shape: (512, 512)


## 4. Particle Tracking

In [5]:
step_name = "Particle Tracking"
start_time = time.time()

# Tracking params
tracking_params = {
    'channels_spots': [1],
    'channels_cytosol': [0],
    'channels_nucleus': [None],
    'threshold_for_spot_detection': 83,
    'yx_spot_size_in_px': 5,
    'z_spot_size_in_px': 2,
    'cluster_radius_nm': 500,
    'min_length_trajectory': 10,
    'maximum_range_search_pixels': 9,
    'memory': 1,
    'link_using_3d_coordinates': True,
    'use_maximum_projection': True,
}

# Ensure mask is bool
binary_mask = (mask_cytosol > 0).astype(bool)

tracker = mi.ParticleTracking(
    image=image_corrected,
    list_voxels=list_voxels,
    masks=binary_mask,
    step_size_in_sec=time_interval_sec,
    **tracking_params
)

list_dataframes, _ = tracker.run()
df_tracking = list_dataframes[0] if list_dataframes else pd.DataFrame()

log_time(step_name, start_time)
print(f"Tracking complete. Found {len(df_tracking)} spots.")

[Particle Tracking] Execution time: 89.5232 seconds
Tracking complete. Found 158186 spots.


## 5. MSD Analysis

In [6]:
step_name = "MSD Analysis"
start_time = time.time()

if not df_tracking.empty and 'particle' in df_tracking.columns:
    microns_per_pixel = pixel_size_xy_nm / 1000
    max_lagtime = 30
    
    # Pre-filter
    min_length = 5
    traj_lengths = df_tracking['particle'].value_counts()
    valid_particles = traj_lengths[traj_lengths >= min_length].index
    df_tracking_filtered = df_tracking[df_tracking['particle'].isin(valid_particles)].copy()
    
    if not df_tracking_filtered.empty:
        msd_analyzer = mi.ParticleMotion(
            trackpy_dataframe=df_tracking_filtered,
            microns_per_pixel=microns_per_pixel,
            step_size_in_sec=time_interval_sec,
            max_lagtime=min(max_lagtime, len(df_tracking_filtered)//2),
            show_plot=False,
            is_3d=False,
            max_fit_points=20
        )
        
        diffusion_coeff, msd_data, _, _, _, _, _ = msd_analyzer.calculate_msd()
        print(f"MSD Analysis complete. D = {diffusion_coeff:.4f} µm²/s")
    else:
        print("No valid trajectories for MSD analysis.")
else:
    print("No tracking data available.")

log_time(step_name, start_time)

MSD Analysis complete. D = 0.0367 µm²/s
[MSD Analysis] Execution time: 10.5227 seconds


10.5226731300354

## Performance Summary

In [7]:
print("\n--- MicroLive Pipeline Performance Summary ---")
total_time = 0
for step, duration in benchmark_results.items():
    print(f"{step}: {duration:.4f} seconds")
    total_time += duration

print(f"\nTotal Execution Time: {total_time:.4f} seconds")

if image is not None:
    data_size_mb = image.nbytes / (1024 * 1024)
    num_frames = image.shape[0]
    print(f"Processed Data Size: {data_size_mb:.2f} MB")
    print(f"Total Frames: {num_frames}")
    print(f"Throughput: {data_size_mb / total_time:.2f} MB/s")

    # Save results to file
    with open('microlive_benchmark_results.txt', 'w') as f:
        f.write("MicroLive Benchmark Results\n")
        f.write("===========================\n")
        f.write(f"OS: {platform.system()} {platform.release()}\n")
        f.write(f"CPU: {platform.processor()}\n")
        f.write(f"Data Size: {data_size_mb:.2f} MB\n")
        f.write(f"Total Time: {total_time:.4f} seconds\n")
        f.write("\nSteps:\n")
        for step, duration in benchmark_results.items():
            f.write(f"{step}: {duration:.4f} s\n")


--- MicroLive Pipeline Performance Summary ---
Data Loading: 9.6535 seconds
Photobleaching Correction: 15.4873 seconds
Cell Segmentation: 0.1372 seconds
Particle Tracking: 89.5232 seconds
MSD Analysis: 10.5227 seconds

Total Execution Time: 125.3238 seconds
Processed Data Size: 7200.00 MB
Total Frames: 360
Throughput: 57.45 MB/s
